# Naive Bayes 분류 예제

이 노트북은 **나이브 베이즈(Naive Bayes) 분류**를 처음 학습하는 사람을 위한 간단한 실습 예제입니다.

이번 예제에서는 scikit-learn에 포함된 **Iris(붓꽃) 데이터셋**을 사용하고,  
연속형 수치 데이터를 다루는 **Gaussian Naive Bayes (`GaussianNB`)** 모델로 붓꽃의 품종을 분류합니다.

## 학습 순서

1. 데이터 불러오기
2. 데이터 구조 확인
3. 독립변수(X)와 종속변수(y) 분리
4. 학습 데이터와 테스트 데이터 분리
5. Gaussian Naive Bayes 모델 학습
6. 예측
7. 모델 성능 평가
8. 예측 확률 확인
9. 새로운 데이터 분류


## 1. 필요한 라이브러리 불러오기

- `pandas`: 데이터를 표 형태로 확인
- `train_test_split`: 학습용/테스트용 데이터 분리
- `GaussianNB`: 가우시안 나이브 베이즈 분류 모델
- `accuracy_score`: 정확도 계산
- `confusion_matrix`: 혼동행렬 확인
- `classification_report`: Precision, Recall, F1-score 확인


In [1]:
import pandas as pd

from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.naive_bayes import GaussianNB
from sklearn.metrics import accuracy_score, confusion_matrix, classification_report

## 2. Iris 데이터셋 불러오기

Iris 데이터셋에는 붓꽃 150개의 정보가 들어 있습니다.

입력값(feature)은 다음 4개입니다.

- `sepal length (cm)` : 꽃받침 길이
- `sepal width (cm)` : 꽃받침 너비
- `petal length (cm)` : 꽃잎 길이
- `petal width (cm)` : 꽃잎 너비

예측하려는 정답(target)은 붓꽃의 품종입니다.

- `0` : setosa
- `1` : versicolor
- `2` : virginica


In [3]:
iris = load_iris()

print("특성 이름:", iris.feature_names)
print("품종 이름:", iris.target_names)
print("데이터 크기:", iris.data.shape)

특성 이름: ['sepal length (cm)', 'sepal width (cm)', 'petal length (cm)', 'petal width (cm)']
품종 이름: ['setosa' 'versicolor' 'virginica']
데이터 크기: (150, 4)


## 3. DataFrame으로 데이터 확인하기

학습 자체에는 NumPy 배열을 바로 사용할 수도 있지만,  
처음 데이터를 살펴볼 때는 DataFrame으로 변환하면 이해하기 쉽습니다.


In [4]:
df = pd.DataFrame(iris.data, columns=iris.feature_names)
df["target"] = iris.target

df.head()

,sepal length (cm),sepal width (cm),petal length (cm),petal width (cm),target
0,5.1,3.5,1.4,0.2,0
1,4.9,3.0,1.4,0.2,0
2,4.7,3.2,1.3,0.2,0
3,4.6,3.1,1.5,0.2,0
4,5.0,3.6,1.4,0.2,0


In [5]:
df.info()

<class 'pandas.DataFrame'>
RangeIndex: 150 entries, 0 to 149
Data columns (total 5 columns):
 #   Column             Non-Null Count  Dtype  
---  ------             --------------  -----  
 0   sepal length (cm)  150 non-null    float64
 1   sepal width (cm)   150 non-null    float64
 2   petal length (cm)  150 non-null    float64
 3   petal width (cm)   150 non-null    float64
 4   target             150 non-null    int64  
dtypes: float64(4), int64(1)
memory usage: 6.0 KB


In [6]:
df["target"].value_counts().sort_index()

target
0    50
1    50
2    50
Name: count, dtype: int64

각 품종이 50개씩 들어 있으므로 클래스의 개수가 동일한 **균형 데이터**임을 확인할 수 있습니다.


## 4. 독립변수(X)와 종속변수(y) 분리

머신러닝 모델에 입력할 데이터와 정답을 분리합니다.

- `X`: 꽃받침/꽃잎의 길이와 너비 → **독립변수(feature)**
- `y`: 붓꽃 품종 → **종속변수(target)**


In [ ]:
X = df.drop("target", axis=1)
y = df["target"]

print("X 크기:", X.shape)
print("y 크기:", y.shape)

## 5. 학습 데이터와 테스트 데이터 분리

전체 데이터를 모두 학습에 사용하면 모델이 처음 보는 데이터에서도 잘 작동하는지 평가하기 어렵습니다.

따라서 데이터를 다음과 같이 나눕니다.

- 80%: 학습 데이터
- 20%: 테스트 데이터

`stratify=y`를 사용하면 각 품종의 비율이 학습/테스트 데이터에서 비슷하게 유지됩니다.


In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=0.2,
    random_state=42,
    stratify=y
)

print("학습 데이터:", X_train.shape)
print("테스트 데이터:", X_test.shape)

## 6. Gaussian Naive Bayes 모델 생성 및 학습

나이브 베이즈는 **베이즈 정리**를 기반으로 각 클래스일 확률을 계산하여  
가장 가능성이 높은 클래스를 선택하는 분류 알고리즘입니다.

`GaussianNB`는 각 feature의 값이 클래스별로 **정규분포(Gaussian distribution)를 따른다고 가정**합니다.

Iris처럼 입력값이 연속형 수치 데이터일 때 간단하게 사용해볼 수 있습니다.


In [ ]:
model = GaussianNB()

model.fit(X_train, y_train)

## 7. 테스트 데이터 예측하기

학습에 사용하지 않았던 `X_test`를 이용하여 품종을 예측합니다.


In [ ]:
y_pred = model.predict(X_test)

result = pd.DataFrame({
    "실제값": y_test.values,
    "예측값": y_pred
})

result.head(10)

## 8. 정확도(Accuracy) 확인

정확도는 전체 테스트 데이터 중 모델이 올바르게 맞힌 데이터의 비율입니다.

예를 들어 정확도가 `0.9667`이라면 약 **96.67%를 정확하게 분류했다**는 뜻입니다.


In [ ]:
accuracy = accuracy_score(y_test, y_pred)

print("Accuracy:", accuracy)
print(f"Accuracy: {accuracy:.2%}")

## 9. Confusion Matrix 확인

혼동행렬은 각 클래스가 실제로 무엇이었고 모델이 무엇이라고 예측했는지를 보여줍니다.

행(row)은 **실제 클래스**, 열(column)은 **예측 클래스**입니다.


In [ ]:
cm = confusion_matrix(y_test, y_pred)

cm_df = pd.DataFrame(
    cm,
    index=[f"실제_{name}" for name in iris.target_names],
    columns=[f"예측_{name}" for name in iris.target_names]
)

cm_df

대각선에 위치한 값은 **정답을 맞힌 개수**입니다.  
대각선 바깥쪽 값은 다른 품종으로 잘못 분류한 개수입니다.


## 10. Classification Report 확인

분류 문제에서는 정확도뿐 아니라 다음 지표도 자주 확인합니다.

- **Precision**: 해당 품종이라고 예측한 것 중 실제로 맞은 비율
- **Recall**: 실제 해당 품종인 데이터 중 모델이 찾아낸 비율
- **F1-score**: Precision과 Recall의 조화평균


In [ ]:
print(
    classification_report(
        y_test,
        y_pred,
        target_names=iris.target_names
    )
)

## 11. 각 클래스의 예측 확률 확인

`predict()`는 최종적으로 선택한 클래스만 반환합니다.

반면 `predict_proba()`를 사용하면 모델이 각 클래스를 얼마나 가능성이 높다고 판단했는지 확인할 수 있습니다.


In [ ]:
probabilities = model.predict_proba(X_test)

prob_df = pd.DataFrame(
    probabilities,
    columns=iris.target_names
)

prob_df.head(10)

예를 들어 어떤 데이터에 대해

- setosa: 0.99
- versicolor: 0.01
- virginica: 0.00

처럼 나온다면 모델은 해당 꽃이 **setosa일 가능성이 가장 높다**고 판단합니다.

각 행의 확률 합은 거의 1입니다.


In [ ]:
prob_df.head().sum(axis=1)

## 12. 새로운 붓꽃 데이터 예측하기

이번에는 직접 만든 새로운 데이터 1개를 모델에 입력해 보겠습니다.

입력 순서는 학습할 때 사용한 feature 순서와 같아야 합니다.

1. 꽃받침 길이
2. 꽃받침 너비
3. 꽃잎 길이
4. 꽃잎 너비


In [ ]:
new_flower = pd.DataFrame(
    [[5.1, 3.5, 1.4, 0.2]],
    columns=iris.feature_names
)

prediction = model.predict(new_flower)[0]
prediction_proba = model.predict_proba(new_flower)[0]

print("예측 클래스 번호:", prediction)
print("예측 품종:", iris.target_names[prediction])

print("\n품종별 예측 확률")
for name, probability in zip(iris.target_names, prediction_proba):
    print(f"{name}: {probability:.4f}")

## 13. 나이브 베이즈에서 'Naive'가 의미하는 것

나이브 베이즈는 계산을 단순하게 만들기 위해 **각 feature가 서로 독립적이라고 가정**합니다.

Iris 데이터에서는 예를 들어

- 꽃잎 길이
- 꽃잎 너비
- 꽃받침 길이
- 꽃받침 너비

가 서로 아무런 영향을 주지 않는다고 단순하게 가정하는 셈입니다.

실제 데이터에서는 feature들이 서로 관련되어 있을 수 있기 때문에 이 가정이 완벽하게 맞는 것은 아닙니다.

그럼에도 불구하고 계산이 빠르고, 데이터에 따라 좋은 성능을 내기 때문에 여러 분류 문제에서 활용됩니다.


## 14. 정리

이번 실습의 핵심 코드는 다음 흐름입니다.

```python
model = GaussianNB()
model.fit(X_train, y_train)

y_pred = model.predict(X_test)

accuracy_score(y_test, y_pred)
```

즉,

**데이터 준비 → 학습/테스트 분리 → 모델 생성 → `fit()` → `predict()` → 평가**

라는 지금까지 사용했던 scikit-learn 분류 모델의 기본 흐름은 그대로 유지됩니다.

### 기억할 핵심

- Naive Bayes는 **확률을 이용하는 분류 알고리즘**입니다.
- `GaussianNB`는 **연속형 수치 데이터**에 사용할 수 있습니다.
- 각 feature가 서로 독립적이라는 단순한 가정을 사용합니다.
- `predict_proba()`를 사용하면 각 클래스에 대한 예측 확률도 확인할 수 있습니다.
